# 02 - LLaMEA Evolutionary Search Pipeline

This notebook is the main entry point for running LLaMEA evolution experiments. It supports:
1. **Single-Problem Evolution**: Quick testing and optimization of an algorithm for a single target BBOB problem.
2. **Multi-Problem Batch Evolution**: Looping over multiple BBOB problem IDs sequentially with automatic database storage.

In [ ]:
# Setup paths and imports
import os
import sys
from functools import partial
from pathlib import Path

# Add project root src/ to sys.path regardless of directory name or launch folder
root_dir = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
src_path = str(root_dir / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from dotenv import load_dotenv

from core.llamea.session import LLaMEASession
from core.orchestrator import run_experiments, EvolutionTask
from core.problems.bbob import BBOBProblem
from core.llamea.prompts import TASK_PROMPT_NOISY, TASK_PROMPT_CLEAN, FORMAT_PROMPT
from infra.llm.client import LLMClient
from infra.storage.sqlite.factory import initialize_sqlite_storage
from infra.storage.code_store.code import CodeRepository

print("LLaMEA Evolution Pipeline initialized with Orchestrator.")

## 1. LLM Provider Connection
Initialize and test connection to the local LLM server configured in your `.env` file.

In [ ]:
env_path = root_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)

llm_provider = os.getenv("LLM_PROVIDER", "local")

llm = LLMClient(llm_provider)
print("=================================================================")
print(f"Initialized Provider: {llm_provider}")
print(f"Model Target:         {getattr(llm, 'model', 'unknown')}")
print("=================================================================")


## 2. Section 1: Single-Problem Evolution
Run LLaMEA optimization algorithm evolution for a single target BBOB problem configuration and persist the results.

In [ ]:
# Configuration for single run
problem_id = 1
dim = 3
instance_id = 1

problem = BBOBProblem(problem_id=problem_id, dim=dim, instance_id=instance_id)

print(f"Target Problem: BBOB {problem_id} ({dim}D, Instance {instance_id})")
print(f"True Optimum: {problem.true_optimum:.4f}")


### Direct Single Session (No Threading / Debug Mode)
Run  directly in the main thread for the clean problem to observe step-by-step logs and catch unhandled exceptions without thread pool wrapping.

In [ ]:
db_repo = initialize_sqlite_storage()
code_repo = CodeRepository()

# Direct non-threaded run for clean problem
clean_session = LLaMEASession(
    problem=problem,
    llm_client=llm,
    db_repo=db_repo,
    code_repo=code_repo,
    budget=100000,
    iterations=5,
    noise_std=0.0,
)

print(f"--- Running Direct LLaMEASession (Experiment #{clean_session._experiment_id}) ---")
clean_result = clean_session.run()

best_sol = clean_result.best_solution
print(f"[OK] Direct clean run complete. Best Solution: {getattr(best_sol, 'name', 'N/A')} | Fitness: {getattr(best_sol, 'fitness', 'N/A')}")


### Execute Single-Problem Evolution

In [ ]:
db_repo = initialize_sqlite_storage()
code_repo = CodeRepository()



tasks = [
    EvolutionTask(
        "clean",
        lambda: LLaMEASession(
            problem=problem,
            llm_client=llm,
            db_repo=db_repo,
            code_repo=code_repo,
            budget=100000,
            iterations=5,
            noise_std=0.0,
        ),
    ),
    EvolutionTask(
        "noisy",
        lambda: LLaMEASession(
            problem=problem,
            llm_client=llm,
            db_repo=db_repo,
            code_repo=code_repo,
            budget=100000,
            iterations=5,
            noise_std=0.5,
        ),
    ),
]

results = run_experiments(tasks=tasks)

best_clean = results["clean"].best_solution
best_noisy = results["noisy"].best_solution

print(f"[OK] Clean evolution complete. Best Solution: {getattr(best_clean, 'name', 'N/A')} | Fitness: {getattr(best_clean, 'fitness', 'N/A')}")
print(f"[OK] Noisy evolution complete. Best Solution: {getattr(best_noisy, 'name', 'N/A')} | Fitness: {getattr(best_noisy, 'fitness', 'N/A')}")
print("[OK] Saved both runs to database and cleaned up checkpoint files.")


## 3. Section 2: Multi-Problem Batch Evolution Loop
Run LLaMEA evolution sequentially across multiple problem IDs. Results are persisted to the database after each problem run.

In [ ]:
# Batch Configuration
batch_problem_ids = [1, 2, 3, 4, 5]
dim = 3
noise_std = 0.05
iterations = 5
max_evaluations = 1000
mode = "noisy" if noise_std > 0.0 else "clean"

print(f"Starting batch evolution loop for problems: {batch_problem_ids} ({dim}D)")


In [ ]:
if llm is None:
    print("Error: Local LLM server must be running to execute batch evolution.")
else:
    # Setup storage with default data/ directory locations
    db_repo = initialize_sqlite_storage()
    code_repo = CodeRepository()
    
    print(f"Starting batch orchestration loop for problems: {batch_problem_ids} ({dim}D)")
    
    for pid in batch_problem_ids:
        print(f"\n>>> Evolving algorithm for BBOB-{pid} concurrently...")
        p = BBOBProblem(problem_id=pid, dim=dim, instance_id=1)
        try:

            tasks = [
                EvolutionTask(
                    "clean",
                    lambda p=p: LLaMEASession(
                        problem=p,
                        llm_client=llm,
                        db_repo=db_repo,
                        code_repo=code_repo,
                        budget=max_evaluations,
                        iterations=iterations,
                        noise_std=0.0,
                    ),
                ),
                EvolutionTask(
                    "noisy",
                    lambda p=p: LLaMEASession(
                        problem=p,
                        llm_client=llm,
                        db_repo=db_repo,
                        code_repo=code_repo,
                        budget=max_evaluations,
                        iterations=iterations,
                        noise_std=0.5,
                    ),
                ),
            ]

            run_experiments(tasks=tasks)
            print(f"Saved concurrent runs for BBOB-{pid} to database.")
        except Exception as e:
            print(f"Failed to evolve algorithm for BBOB-{pid}: {e}")
            
    print("\nBatch evolution loop processing complete.")
